# API Client Example

This notebook demonstrates how to load a campaign registry from `config/campaigns_dataset.csv`, download every listed campaign with `api.client`, persist the sensor CSV payloads under `data/campaigns/`, inspect the canonical `metadata.csv` for one downloaded campaign, and plot representative stored spectra.

**System boundary**
- Domain/boundary: `api.client` owns HTTP pagination, TLS settings, payload normalization, `metadata.csv` materialization, and sensor CSV persistence.
- Application/orchestration: this notebook validates the campaign registry, triggers one download per listed campaign, summarizes the materialized outputs, inspects one representative campaign, and renders figures.
- Side effects: HTTPS requests to the remote API, CSV writes under `data/`, and notebook plots.
- Pattern choice: the notebook stays thin so the boundary behavior remains testable outside Jupyter.


In [ ]:
from __future__ import annotations

import importlib
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    """Return the repository root regardless of the notebook launch directory."""

    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "measurement_calibration").exists() and (
            candidate / "data"
        ).exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root from the current notebook session"
    )


def load_campaign_dataset(
    csv_path: Path,  # Campaign registry path under config/
) -> pd.DataFrame:  # Validated campaign download plan
    """Load and validate the campaign registry used for batch downloads.

    Parameters
    ----------
    csv_path:
        CSV file that declares one `campaign_label` and `campaign_id` pair per
        row. The historical file currently ships with the misspelled header
        `capaign_label`, which this loader normalizes explicitly.

    Returns
    -------
    pd.DataFrame
        Normalized registry with the columns `campaign_label` and
        `campaign_id`, sorted in the original CSV order.

    Raises
    ------
    FileNotFoundError
        If the registry path does not exist.
    ValueError
        If the CSV is empty, misses required columns, contains duplicated
        campaigns, or contains invalid labels/identifiers.
    """

    if not csv_path.exists():
        raise FileNotFoundError(f"Campaign dataset CSV does not exist: {csv_path}")

    dataset_frame = pd.read_csv(csv_path)
    dataset_frame = dataset_frame.rename(columns={"capaign_label": "campaign_label"})
    dataset_frame = dataset_frame.rename(columns=lambda column: str(column).strip())
    required_columns = ("campaign_label", "campaign_id")
    missing_columns = [
        column for column in required_columns if column not in dataset_frame.columns
    ]
    if missing_columns:
        raise ValueError(
            "Campaign dataset CSV is missing required columns: "
            f"{missing_columns}"
        )

    dataset_frame = dataset_frame.loc[:, list(required_columns)].copy()
    if dataset_frame.empty:
        raise ValueError("Campaign dataset CSV does not contain any campaigns")

    # Normalize the caller-owned labels before issuing any network requests.
    dataset_frame["campaign_label"] = dataset_frame["campaign_label"].astype(str).str.strip()
    if (dataset_frame["campaign_label"] == "").any():
        raise ValueError("Campaign dataset CSV contains an empty campaign_label")

    campaign_ids = pd.to_numeric(dataset_frame["campaign_id"], errors="coerce")
    if campaign_ids.isna().any():
        raise ValueError("Campaign dataset CSV contains a non-numeric campaign_id")
    if (campaign_ids <= 0).any():
        raise ValueError("Campaign dataset CSV contains a non-positive campaign_id")
    dataset_frame["campaign_id"] = campaign_ids.astype(np.int64)

    if dataset_frame.duplicated(subset=["campaign_label"]).any():
        raise ValueError("Campaign dataset CSV contains duplicated campaign_label values")
    if dataset_frame.duplicated(subset=["campaign_id"]).any():
        raise ValueError("Campaign dataset CSV contains duplicated campaign_id values")

    return dataset_frame


def select_analysis_campaign_result(
    download_results: list[client.CampaignDownloadResult],  # Materialized batch results
) -> client.CampaignDownloadResult:  # One campaign with saved sensor CSVs
    """Pick one successfully downloaded campaign for downstream inspection."""

    for download_result in download_results:
        if download_result.saved_csv_paths:
            return download_result
    raise ValueError("No downloaded campaign produced saved sensor CSV files")


def validate_row_indices(
    n_records: int,  # Number of rows available in every loaded sensor CSV
    requested_indices: tuple[int, ...],  # Record indices requested for plotting
) -> tuple[int, ...]:  # Valid row indices in the original requested order
    """Keep only row indices that are valid for every loaded sensor CSV."""

    valid_indices = tuple(
        index for index in requested_indices if 0 <= index < n_records
    )
    if not valid_indices:
        raise ValueError("No requested row indices exist in the downloaded data")
    return valid_indices


def build_frequency_axis_mhz(
    start_freq_hz: float,  # Lower band edge from one stored measurement [Hz]
    end_freq_hz: float,  # Upper band edge from one stored measurement [Hz]
    n_bins: int,  # Number of PSD bins in the stored measurement
) -> np.ndarray:  # Frequency axis in MHz aligned with the PSD vector
    """Build an evenly spaced plotting axis for one PSD vector."""

    if n_bins <= 0:
        raise ValueError("n_bins must be positive")
    return np.linspace(start_freq_hz / 1.0e6, end_freq_hz / 1.0e6, n_bins)


def select_representative_row_indices(
    n_records: int,  # Number of aligned measurements available across sensors
) -> tuple[int, ...]:  # First, midpoint, and last indices without duplicates
    """Select representative record indices for campaign overlay plots."""

    if n_records <= 0:
        raise ValueError("n_records must be positive")

    ordered_indices: list[int] = []
    for index in (0, n_records // 2, n_records - 1):
        if index not in ordered_indices:
            ordered_indices.append(index)
    return tuple(ordered_indices)


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import api.client as client

client = importlib.reload(client)
pd.options.display.max_colwidth = 120


## Configure The Batch Download

The notebook reads `config/campaigns_dataset.csv` as the batch download plan. Each row declares one campaign label/id pair, and the client still adapts dynamically when a specific campaign is only available for a subset of the `Node1..Node10` sensor network.


In [ ]:
campaign_dataset_csv_path = REPO_ROOT / "config" / "campaigns_dataset.csv"
campaign_dataset_frame = load_campaign_dataset(campaign_dataset_csv_path)
sensor_mac_by_label = client.SENSOR_NETWORK_MAC_BY_LABEL
output_root = REPO_ROOT / client.CAMPAIGNS_DATA_DIR

# The deployed API currently requires verify_tls=False in this environment.
api_client = client.MeasurementApiClient(
    client.MeasurementApiConfig(verify_tls=False, timeout_s=30.0, page_size=5000)
)

display(
    pd.DataFrame(
        [
            {
                "campaign_dataset_csv": str(campaign_dataset_csv_path.relative_to(REPO_ROOT)),
                "n_campaigns": len(campaign_dataset_frame),
                "output_root": str(output_root.relative_to(REPO_ROOT)),
                "n_sensors": len(sensor_mac_by_label),
            }
        ]
    )
)
display(campaign_dataset_frame)
display(
    pd.DataFrame(
        [
            {"sensor_label": sensor_label, "mac": mac_address}
            for sensor_label, mac_address in sensor_mac_by_label.items()
        ]
    ).sort_values("sensor_label")
)


## Download And Persist CSV Files

This cell iterates over every row declared in `config/campaigns_dataset.csv`, creates `data/campaigns/<campaign>/`, writes one canonical `metadata.csv` per campaign, and writes one measurement CSV per retained sensor using the repository's acquisition schema. Sensors that are unavailable for a campaign are reported explicitly instead of aborting the full batch.


In [ ]:
download_results: list[client.CampaignDownloadResult] = []
download_summary_rows: list[dict[str, object]] = []
skipped_sensor_rows: list[dict[str, object]] = []

# Keep the batch loop explicit so per-campaign failures remain attributable.
for campaign_row in campaign_dataset_frame.itertuples(index=False):
    download_result = api_client.download_campaign_csvs(
        campaign_label=campaign_row.campaign_label,
        campaign_id=int(campaign_row.campaign_id),
        output_root=output_root,
        drop_missing_pxx=True,
        skip_missing_sensors=True,
    )
    download_results.append(download_result)

    metadata_path = download_result.metadata_csv_path
    download_summary_rows.append(
        {
            "campaign_label": download_result.campaign_label,
            "campaign_id": download_result.campaign_id,
            "metadata_csv_path": (
                ""
                if metadata_path is None
                else str(metadata_path.relative_to(REPO_ROOT))
            ),
            "n_sensor_csvs": len(download_result.saved_csv_paths),
            "n_skipped_sensors": len(download_result.skipped_sensors),
            "output_dir": str(download_result.output_dir.relative_to(REPO_ROOT)),
        }
    )

    for sensor_label, reason in sorted(download_result.skipped_sensors.items()):
        skipped_sensor_rows.append(
            {
                "campaign_label": download_result.campaign_label,
                "campaign_id": download_result.campaign_id,
                "sensor_label": sensor_label,
                "reason": reason,
            }
        )

if not download_results:
    raise ValueError("The campaign dataset did not produce any download requests")

download_summary_frame = pd.DataFrame(download_summary_rows).sort_values(
    ["campaign_id", "campaign_label"]
)
display(download_summary_frame)

display(
    pd.DataFrame(
        [
            {
                "n_campaigns_requested": len(download_results),
                "n_campaigns_with_sensor_csvs": int(
                    sum(bool(result.saved_csv_paths) for result in download_results)
                ),
                "n_total_sensor_csvs": int(
                    sum(len(result.saved_csv_paths) for result in download_results)
                ),
                "n_total_skipped_sensors": int(
                    sum(len(result.skipped_sensors) for result in download_results)
                ),
            }
        ]
    )
)

if skipped_sensor_rows:
    display(pd.DataFrame(skipped_sensor_rows).sort_values(["campaign_id", "sensor_label"]))


## Inspect One Downloaded Campaign

The batch step downloads every campaign from the registry. The remaining cells keep one representative campaign in focus so the notebook still offers a compact path for inspecting `metadata.csv`, reloading stored sensor CSV files, and plotting spectra.


In [ ]:
analysis_download_result = select_analysis_campaign_result(download_results)
analysis_campaign_label = analysis_download_result.campaign_label
analysis_campaign_id = analysis_download_result.campaign_id
stored_paths = analysis_download_result.saved_csv_paths
metadata_path = analysis_download_result.metadata_csv_path

if metadata_path is None:
    raise ValueError(
        "The selected analysis campaign does not include a materialized metadata.csv"
    )

metadata_frame = pd.read_csv(metadata_path)
frames_by_sensor = client.load_measurement_frames(stored_paths)
summary_rows: list[dict[str, object]] = []

if not frames_by_sensor:
    raise ValueError("The selected analysis campaign produced no saved sensor CSV files")

# Summarize the representative campaign before plotting individual spectra.
for sensor_label, frame in frames_by_sensor.items():
    first_row = frame.iloc[0]
    summary_rows.append(
        {
            "sensor_label": sensor_label,
            "records": len(frame),
            "frequency_start_mhz": float(first_row["start_freq_hz"]) / 1.0e6,
            "frequency_end_mhz": float(first_row["end_freq_hz"]) / 1.0e6,
            "n_bins": int(np.asarray(first_row["pxx"]).size),
            "first_timestamp_ms": int(first_row["timestamp"]),
        }
    )
summary_frame = pd.DataFrame(summary_rows).sort_values("sensor_label")

display(
    pd.DataFrame(
        [
            {
                "analysis_campaign_label": analysis_campaign_label,
                "analysis_campaign_id": analysis_campaign_id,
                "metadata_csv_path": str(metadata_path.relative_to(REPO_ROOT)),
                "n_sensor_csvs": len(stored_paths),
            }
        ]
    )
)
display(metadata_frame)
display(
    pd.DataFrame(
        [
            {
                "sensor_label": sensor_label,
                "csv_path": str(csv_path.relative_to(REPO_ROOT)),
            }
            for sensor_label, csv_path in stored_paths.items()
        ]
    ).sort_values("sensor_label")
)
display(summary_frame)


## Campaign Analysis

This section summarizes the time span and coarse PSD statistics for each sensor in the representative campaign selected from the batch download results.


In [ ]:
analysis_rows: list[dict[str, object]] = []

# Reduce each sensor to stable campaign-level descriptors that are easy to compare.
for sensor_label, frame in sorted(frames_by_sensor.items()):
    timestamps_ms = frame["timestamp"].dropna().astype(np.int64)
    pxx_arrays = [np.asarray(pxx, dtype=np.float64) for pxx in frame["pxx"]]
    pxx_lengths = np.asarray([pxx.size for pxx in pxx_arrays], dtype=np.int64)
    mean_band_power_dbm = np.asarray([float(np.mean(pxx)) for pxx in pxx_arrays])
    noise_floor_dbm = np.asarray(
        [float(np.percentile(pxx, 10.0)) for pxx in pxx_arrays],
        dtype=np.float64,
    )

    start_timestamp_ms = int(timestamps_ms.min())
    end_timestamp_ms = int(timestamps_ms.max())
    analysis_rows.append(
        {
            "sensor_label": sensor_label,
            "records": len(frame),
            "timestamp_start_utc": pd.to_datetime(
                start_timestamp_ms,
                unit="ms",
                utc=True,
            ),
            "timestamp_end_utc": pd.to_datetime(
                end_timestamp_ms,
                unit="ms",
                utc=True,
            ),
            "time_span_min": (end_timestamp_ms - start_timestamp_ms) / 60_000.0,
            "median_n_bins": int(np.median(pxx_lengths)),
            "mean_band_power_dbm": float(np.mean(mean_band_power_dbm)),
            "mean_noise_floor_dbm": float(np.mean(noise_floor_dbm)),
        }
    )

campaign_analysis_frame = pd.DataFrame(analysis_rows).sort_values("sensor_label")
display(
    pd.DataFrame(
        [
            {
                "analysis_campaign_label": analysis_campaign_label,
                "analysis_campaign_id": analysis_campaign_id,
            }
        ]
    )
)
display(campaign_analysis_frame)


## Plot Stored Spectra

The figure overlays the same record index across all saved sensor CSV files for the representative campaign. If a requested row index does not exist for every sensor, it is skipped explicitly.


In [ ]:
min_records = min(len(frame) for frame in frames_by_sensor.values())
row_indices = validate_row_indices(
    min_records,
    select_representative_row_indices(min_records),
)

fig, axes = plt.subplots(
    nrows=len(row_indices),
    ncols=1,
    figsize=(18, 4.5 * len(row_indices)),
    sharex=True,
)
if len(row_indices) == 1:
    axes = [axes]

for ax, row_index in zip(axes, row_indices):
    for sensor_label, frame in sorted(frames_by_sensor.items()):
        row = frame.iloc[row_index]
        pxx_db = np.asarray(row["pxx"], dtype=np.float64)

        # Each stored row carries its own band edges, so build the plotting
        # axis from the persisted metadata instead of assuming a fixed band.
        frequency_mhz = build_frequency_axis_mhz(
            start_freq_hz=float(row["start_freq_hz"]),
            end_freq_hz=float(row["end_freq_hz"]),
            n_bins=pxx_db.size,
        )
        ax.plot(frequency_mhz, pxx_db, label=sensor_label, linewidth=1.0, alpha=0.85)

    ax.set_title(f"{analysis_campaign_label} PSD Overlay, row {row_index}")
    ax.set_ylabel("Power [dBm]")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="best")

axes[-1].set_xlabel("Frequency [MHz]")
plt.tight_layout()
plt.show()
